# Portuguese Bank Marketing - Data Analysis & Predictive Modeling

This notebook contains:
- **Task 1:** Exploratory Data Analysis (EDA)
- **Task 2:** Predictive modeling (who will subscribe)
- **Task 3:** Actionable suggestions for the marketing team

It was auto-generated from our analysis pipeline.

In [24]:
import pandas as pd

# Load dataset
df = pd.read_excel("bank_full_cleaned.xlsx")
df.head()

,Age,Job,Marital,Education,Default,Balance,Housing,Loan,Contact,Day,Month,Duration (s),Campaign,Pdays,Previous,Poutcome,Subscribed
0,58,Management,Married,Tertiary,No,2143,Yes,No,Unknown,5,May,261,1,-1,0,Unknown,No
1,44,Technician,Single,Secondary,No,29,Yes,No,Unknown,5,May,151,1,-1,0,Unknown,No
2,33,Entrepreneur,Married,Secondary,No,2,Yes,Yes,Unknown,5,May,76,1,-1,0,Unknown,No
3,47,Blue-collar,Married,Unknown,No,1506,Yes,No,Unknown,5,May,92,1,-1,0,Unknown,No
4,33,Unknown,Single,Unknown,No,1,No,No,Unknown,5,May,198,1,-1,0,Unknown,No


## Task 1 - Exploratory Data Analysis

I inspect class balance, key numeric/categorical effects, and drop leakage-prone duration.

In [26]:
df['Subscribed'].value_counts(normalize=True)

Subscribed
No     0.883015
Yes    0.116985
Name: proportion, dtype: float64

## Task 2 - Predictive Modeling

We compare Logistic Regression, Random Forest, and Gradient Boosting with one-hot + scaling pipeline.

Duration was excluded to avoid data leakage.

### Suggestions for Bank Marketing Team

1. Call smartly, not too much
   Customers don’t like too many follow-up calls. After some tries, chances of success go down. So keep the number of calls limited and focus on quality talks.

2. Choose the right time
   Some months and days have better response rates. Try to contact more customers during those periods to improve results.

3. Target the right people
   People with higher account balance and certain age groups are more likely to invest. Focus on these groups for higher success.

4. Cross-sell carefully

   a. Customers with no loan see deposits as a safe starting point.
   b. Customers with loans like deposits when shown as a safety fund or extra support.

5. Personalise the approach
   Job type and education level affect decisions. For example, entrepreneurs care more about liquidity, while salaried workers prefer security and ease.

6. Follow up with warm leads
   People who already showed interest in past campaigns or were contacted recently are more likely to say yes. Give them priority.

7. Make offers flexible
   Provide deposit options like short-term and medium-term (laddered deposits). Allow auto-renewal with easy opt-out. Show deposits as a tool for goals like children’s education, travel, or emergencies.

8. Improve the sales script
   Keep calls short and simple. First talk about customer goals and safety, then explain the interest rates. Long calls may bore customers.

9. Keep learning from data
   Customer behaviour changes with time. Re-train the predictive model regularly (for example every 3 months) to keep improving


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [36]:
# Prepare data
#df = df.dropna(subset=['Subscribed'])  
#X = df.drop(columns=['Subscribed','Duration (s)'], errors='ignore')
#y = df['Subscribed'].map({'yes':1,'no':0})
# Clean target column robustly
df['Subscribed'] = df['Subscribed'].astype(str).str.strip().str.lower()

# Map to binary
df = df[df['Subscribed'].isin(['yes','no'])]   # keep only rows with yes/no
df['Subscribed'] = df['Subscribed'].map({'yes':1, 'no':0})

# Now safe
X = df.drop(columns=['Subscribed','Duration (s)'], errors='ignore')
y = df['Subscribed']


In [38]:
cat_cols = [c for c in X.columns if X[c].dtype == 'object']
num_cols = [c for c in X.columns if c not in cat_cols]

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols)
])

In [42]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
    "GradientBoosting": GradientBoostingClassifier(random_state=42)
}


In [44]:

for name, clf in models.items():
    pipe = Pipeline([('pre', preprocess),('model', clf)])
    pipe.fit(X_train,y_train)
    y_proba = pipe.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test,y_proba)
    print(name, "ROC AUC:", auc)

LogisticRegression ROC AUC: 0.7717487775401183
RandomForest ROC AUC: 0.7925662247148185
GradientBoosting ROC AUC: 0.8021732028271346


In [46]:
# taking the help of chatgpt via promt for more efficent work all the observation is witen by me.